# DEA Coastlines summary plots

Plan for Torres Strait Authority State of Environment delivery:

1. Provide a tabluar summary (divided up spatially however we like i.e. whole TS region, main island centers etc) with two analyses per epoch (5 year SoE reporting period):
    1. Long term coastal change summary (1988 to baseline year e.g SoE reporting years)
    2. Short term coastal change summary (preceding 5 years to the SoE baseline year)
1. Instructions on how to download individual coastlines temporal profiles directly from DEA maps

This requires reconfiguring the analysis to re-develop the calculations for new baselines. As a prototype, we can deliver a short and long term summary with a 2024 baseline. I'll need to rework things to use earlier baselines.

We can explore whether figures are appropriate to support this. We might be able to spatially separate the islands into east/west aspects to look at oceanographic influences on coastal erosion. This is a later job :)

In [245]:
# cd ../'Tools'

In [246]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from dea_tools.coastal import get_coastlines

# Load DEA Coastlines code
# import coastlines.raster
# import coastlines.vector

from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from scipy.stats import linregress

In [247]:
# Load Coastlines functions

# These are edited functions from dea-coastlines GItHub repo vectors.py

def outlier_mad(points, thresh=3.5):
    """
    Use robust Median Absolute Deviation (MAD) outlier detection
    algorithm to detect outliers. Returns a boolean array with True if
    points are outliers and False otherwise.

    Parameters:
    -----------
    points :
        An n-observations by n-dimensions array of observations
    thresh :
        The modified z-score to use as a threshold. Observations with a
        modified z-score (based on the median absolute deviation) greater
        than this value will be classified as outliers.

    Returns:
    --------
    mask :
        A n-observations-length boolean array.

    References:
    ----------
    Source: https://github.com/joferkington/oost_paper_code/blob/master/utilities.py

    Boris Iglewicz and David Hoaglin (1993), "Volume 16: How to Detect and
    Handle Outliers", The ASQC Basic References in Quality Control:
    Statistical Techniques, Edward F. Mykytka, Ph.D., Editor.
    """
    if len(points.shape) == 1:
        points = points[:, None]
    median = np.median(points, axis=0)
    diff = np.sum((points - median) ** 2, axis=-1)
    diff = np.sqrt(diff)
    med_abs_deviation = np.median(diff)

    modified_z_score = 0.6745 * diff / med_abs_deviation

    return modified_z_score > thresh

def change_regress(
    y_vals,
    x_vals,
    x_labels,
    threshold=3.5,
    detrend_params=None,
    slope_var="slope",
    interc_var="intercept",
    pvalue_var="pvalue",
    stderr_var="stderr",
    outliers_var="outliers",
):
    """
    For a given row in a `pandas.DataFrame`, apply linear regression to
    data values (as y-values) and a corresponding sequence of x-values,
    and return 'slope', 'intercept', 'pvalue', and 'stderr' regression
    parameters.

    Before computing the regression, outliers are identified using a
    robust Median Absolute Deviation (MAD) outlier detection algorithm,
    and excluded from the regression. A list of these outliers will be
    recorded in the output 'outliers' variable.

    Parameters:
    -----------
    x_vals, y_vals : list of numeric values, or nd.array
        A sequence of values to use as the x and y variables
    x_labels : list
        A sequence of strings corresponding to each value in `x_vals`.
        This is used to label any observations that are flagged as
        outliers (often, this can simply be set to the same list
        provided to `x_vals`).
    threshold : float, optional
        The modified z-score to use as a threshold for detecting
        outliers using the MAD algorithm. Observations with a modified
        z-score (based on the median absolute deviation) greater
        than this value will be classified as outliers.
    detrend_params : optional
        Not currently used
    slope, interc_var, pvalue_var, stderr_var : strings, optional
        Strings giving the names to use for each of the output
        regression variables.
    outliers_var : string, optional
        String giving the name to use for the output outlier variable.

    Returns:
    --------
    mask :
        A `pandas.Series` containing regression parameters and lists
        of outliers.

    """

    # Drop invalid NaN rows
    xy_df = np.vstack([x_vals, y_vals]).T
    valid_bool = ~np.isnan(xy_df).any(axis=1)
    xy_df = xy_df[valid_bool]
    valid_labels = x_labels[valid_bool]

    # Check if we have enough data points after removing NaNs (minimum 2)
    if len(xy_df) < 2:
        # Create string of all outliers and invalid NaN rows
        outlier_set = set(x_labels) - set(valid_labels)
        outlier_str = " ".join(map(str, sorted(outlier_set)))
        
        # Return NaN values if insufficient data
        results_dict = {
            slope_var: np.nan,
            interc_var: np.nan,
            pvalue_var: np.nan,
            stderr_var: np.nan,
            outliers_var: outlier_str,
        }
        return pd.Series(results_dict)
    
    # If detrending parameters are provided, apply these to the data to
    # remove the trend prior to running the regression
    if detrend_params:
        xy_df[:, 1] = xy_df[:, 1] - (
            detrend_params[0] * xy_df[:, 0] + detrend_params[1]
        )

    # Remove outliers using MAD
    outlier_bool = outlier_mad(xy_df, thresh=threshold)
    # outlier_bool = outlier_ransac(xy_df)
    xy_df = xy_df[~outlier_bool]
    valid_labels = valid_labels[~outlier_bool]

    # Create string of all outliers and invalid NaN rows
    outlier_set = set(x_labels) - set(valid_labels)
    outlier_str = " ".join(map(str, sorted(outlier_set)))

    # Check again after outlier removal
    if len(xy_df) < 2:
        # Return NaN values if insufficient data
        results_dict = {
            slope_var: np.nan,
            interc_var: np.nan,
            pvalue_var: np.nan,
            stderr_var: np.nan,
            outliers_var: outlier_str,
        }
        return pd.Series(results_dict)
    
    # Compute linear regression
    lin_reg = linregress(x=xy_df[:, 0], y=xy_df[:, 1])

    # Return slope, p-values and list of outlier years excluded from regression
    results_dict = {
        slope_var: np.round(lin_reg.slope, 3),
        interc_var: np.round(lin_reg.intercept, 3),
        pvalue_var: np.round(lin_reg.pvalue, 3),
        stderr_var: np.round(lin_reg.stderr, 3),
        outliers_var: outlier_str,
    }

    return pd.Series(results_dict)

def calculate_regressions(points_gdf1,
                         start,
                         end):
    """
    For each rate of change point along the baseline annual coastline,
    compute linear regression rates of change against both time and
    climate indices.

    Regressions are computed after removing outliers to ensure robust
    results.

    Parameters:
    -----------
    points_gdf : geopandas.GeoDataFrame
        A `geopandas.GeoDataFrame` containing rates of change points
        with 'dist_*' annual movement/distance data.
    start  :  integer (optional)
        A 4 digit integer of the year you wish to begin the epoch calculation.
        For example, the baseline is set as 2020 from the annual_movements function.
        Set `start` = 2015 to calculate rate_change points for the 5 year window
        between 2015 to 2020.
        All years before 'start' are excluded from the rate change calculation.
        Defaults to none, setting the oldest year as the start of the epoch.
    end  :  integer (optional)
        An integer of the year used as baseline for rate change calculations.
        Only used in the variable naming in the returned geopandas.GeoDataFrame

    Returns:
    --------
    points_gdf : geopandas.GeoDataFrame
        A `geopandas.GeoDataFrame` containing rates of change points
        with additional attribute columns:

            'rate_*':  Slope of the regression
            'sig_*':   Significance of the regression
            'se_*':    Standard error of the  regression
            'outl_*':  A list of any outlier years excluded from the
                       regression
    """
    # Copy the input geodataframe
    points_gdf = points_gdf1.copy(deep=True)
    
    # Restrict data to years in datasets
    dist_years = points_gdf.columns[points_gdf.columns.str.contains("dist_")]
    x_years = dist_years.str.replace("dist_", "").astype(int)
    if start:
        x = x_years.where(x_years>=start).dropna().astype('int')
        dist_years = dist_years[-len(x):]
        print(dist_years)
        points_subset = points_gdf[dist_years].copy()
    else:
        points_subset = points_gdf[dist_years].copy()
    
    # Compute coastal change rates by linearly regressing annual
    # movements vs. time
    if start:
        rate_out = points_subset.apply(
            lambda row: change_regress(
                y_vals=row.values.astype(float), x_vals=x, x_labels=x
            ),
            axis=1,
            )
        points_gdf[[f"rate_time_{start}-{end}", f"incpt_time_{start}-{end}", f"sig_time_{start}-{end}", f"se_time_{start}-{end}", f"outl_time_{start}-{end}"]] = (
            rate_out
            )
    
    else:
        rate_out = points_subset.apply(
            lambda row: change_regress(
                y_vals=row.values.astype(float), x_vals=x_years, x_labels=x_years
            ),
            axis=1,
            )
        points_gdf[["rate_time", "incpt_time", "sig_time", "se_time", "outl_time"]] = (
                rate_out
            )
    
    # Copy slope and intercept into points_subset so they can be
    # used to temporally de-trend annual distances
    points_subset[["slope", "intercept"]] = rate_out[["slope", "intercept"]]
    
    # Custom sorting
    if start:
        reg_cols = [f"rate_time_{start}-{end}", f"sig_time_{start}-{end}", f"se_time_{start}-{end}", f"outl_time_{start}-{end}"]
    else:
        reg_cols = ["rate_time", "sig_time", "se_time", "outl_time"]
    
    
    return points_gdf.loc[
        :, [*reg_cols, *dist_years, "angle_mean", "angle_std", "geometry"]
    ]

# Edited functions from dea_tools.coastal
def get_coastlines1(response, layer, bbox: tuple, crs="EPSG:4326") -> gpd.GeoDataFrame:
    """
    Load DEA Coastlines annual shorelines or rates of change points data
    for a provided bounding box using WFS.

    For a full description of the DEA Coastlines dataset, refer to the
    official Geoscience Australia product description:
    /data/product/dea-coastlines

    Parameters
    ----------
    bbox : (xmin, ymin, xmax, ymax), or geopandas object
        Bounding box expressed as a tutple. Alternatively, a bounding
        box can be automatically extracted by suppling a
        geopandas.GeoDataFrame or geopandas.GeoSeries.
    crs : str, optional
        Optional CRS for the bounding box. This is ignored if `bbox`
        is provided as a geopandas object.
    layer : str, optional
        Which DEA Coastlines layer to load. Options include the annual
        shoreline vectors ("shorelines_annual") and the rates of change
        points ("rates_of_change"). Defaults to "shorelines_annual".
    response  :  str
        A file directory path to the annual rates of change DEA Coastlines
        dataset

    Returns
    -------
    gpd.GeoDataFrame
        A GeoDataFrame containing shoreline or point features and
        associated metadata.
    """

    # If bbox is a geopandas object, convert to bbox
    try:
        crs = str(bbox.crs)
        bbox = bbox.total_bounds
    except:
        pass

    # # Query WFS
    # wfs = WebFeatureService(url=WFS_ADDRESS, version="1.1.0")
    # layer_name = f"dea:{layer}"
    # response = wfs.getfeature(
    #     typename=layer_name,
    #     bbox=tuple(bbox) + (crs,),
    #     outputFormat="json",
    # )

    # Load data as a geopandas.GeoDataFrame
    coastlines_gdf = gpd.read_file(response,layer=layer)

    # Clip to extent of bounding box
    extent = gpd.GeoSeries(box(*bbox), crs=crs).to_crs(coastlines_gdf.crs)
    coastlines_gdf = coastlines_gdf.clip(extent)

    # # Optionally drop WMS-specific columns
    # if drop_wms:
    #     coastlines_gdf = coastlines_gdf.loc[:, ~coastlines_gdf.columns.str.contains("wms_")]

    return coastlines_gdf


## Load spatial data and most recent annual change rates

From DEA Coastlines Knowledge Hub [Specifications](https://knowledge.dea.ga.gov.au/data/product/dea-coastlines/?tab=specifications#layers)

	
||Type|Units|Description|
|---|---|---|---|
|uid|String|-|A unique geohash identifier for each point.|
|rate_time|Float|Metres per year|Annual rates of change (in metres per year) calculated by linearly regressing annual shoreline distances against time (excluding outliers). Negative values indicate retreat and positive values indicate growth.|
|sig_time|Float|P-value|Significance (p-value) of the linear relationship between annual shoreline distances and time. Small values (e.g. p-value < 0.01 or 0.05) may indicate a coastline is undergoing consistent coastal change through time.|
|se_time|Float|Metres|Standard error (in metres) of the linear relationship between annual shoreline distances and time. This can be used to generate confidence intervals around the rate of change given by rate_time, e.g. 95% confidence interval = se_time x 1.96|
|outl_time|String|-|Individual annual shoreline are noisy estimators of coastline position that can be influenced by environmental conditions (e.g. clouds, breaking waves, sea spray) or modelling issues (e.g. poor tidal modelling results or limited clear satellite observations). To obtain reliable rates of change, outlier shorelines are excluded using a robust Median Absolute Deviation outlier detection algorithm, and recorded in this column.|
|dist_1990, dist_1991, etc|Float|Metres|Annual shoreline distances (in metres) relative to the most recent baseline shoreline. Negative values indicate that an annual shoreline was located inland of the baseline shoreline. By definition, the most recent baseline column will always have a distance of 0 m.|
|angle_mean, angle_std|Integer|Degrees|The mean angle and standard deviation between the baseline point to all annual shorelines. This data is used to calculate how well shorelines fall along a consistent line; high angular standard deviation indicates that derived rates of change are unlikely to be correct.|
|valid_obs|Integer|-|The total number of valid (i.e. non-outliers, non-missing) annual shoreline observations.|
valid_span|Integer|Years|The maximum number of years between the first and last valid annual shoreline.|
|sce|Float|Metres|Shoreline Change Envelope (SCE). A measure of the maximum change or variability across all annual shorelines, calculated by computing the maximum distance between any two annual shorelines (excluding outliers). This statistic excludes sub-annual shoreline variability.|
|nsm|Float|Metres|Net Shoreline Movement (NSM). The distance between the oldest (1988) and most recent annual shoreline (excluding outliers). Negative values indicate the coastline retreated between the oldest and most recent shoreline; positive values indicate growth. This statistic does not reflect sub-annual shoreline variability, so will underestimate the full extent of variability at any given location.|
|max_year|Integer|Date|The year that annual shorelines were at their maximum (i.e. located furthest towards the ocean), excluding outliers. This statistic excludes sub-annual shoreline variability.|
|min_year|Integer|Date|The year that annual shorelines were at their minimum (i.e. located furthest inland), excluding outliers. This statistic excludes sub-annual shoreline variability.|
|certainty|String|-|A column providing important data quality flags for each point in the dataset. For more information, see the Quality tab.|
|id_primary|String|-|The name of the point’s Primary sediment compartment from the Australian Coastal Sediment Compartments framework.|

In [248]:
# Set study area from vector file

## Use regions_gdf for mainland polygons
regions_gdf1 = (
    gpd.read_file(
        '~/gdata1/projects/coastal/SoE/Torres_Strait_2025/Torres Strait Protected Zone.shp' ## Greater Torres Strait area
        
    )
    # .set_index("Descriptio")
    # .set_index("layer")
    # .set_index('id')
)
# regions_gdf

regions_gdf2 = (
    gpd.read_file(
        '~/gdata1/projects/coastal/SoE/Torres_Strait_2025/TS Island Coastline Analysis boundaries.shp' ## Greater Torres Strait area
        
    )
    # .set_index("Descriptio")
    # .set_index("layer")
    # .set_index('id')
)

regions_gdf = pd.concat([regions_gdf2, gpd.GeoDataFrame(regions_gdf1.to_crs(regions_gdf2.crs))])
regions_gdf.loc[regions_gdf['name'].isna(), 'name'] = 'Torres Strait'

regions_gdf
# regions_gdf = regions_gdf1.to_crs(regions_gdf2.crs).sjoin(regions_gdf2, how='inner', predicate='intersects')

study_area = regions_gdf
# study_area.geometry[0]
study_area

,Id,name,geometry
0,0,Boigu,"POLYGON ((142.21655 -9.22601, 142.22660 -9.226..."
1,0,Saibai,"POLYGON ((142.79838 -9.36651, 142.80358 -9.370..."
2,0,Dauan,"POLYGON ((142.55273 -9.42149, 142.55029 -9.423..."
3,0,Waral Kawa,"POLYGON ((141.57438 -9.51246, 141.57527 -9.514..."
4,0,Awail Kawa,"POLYGON ((141.56899 -9.60609, 141.57193 -9.606..."
5,0,Mabuiag,"POLYGON ((142.19717 -9.95251, 142.19421 -9.953..."
6,0,Badu,"POLYGON ((142.17142 -10.06306, 142.17265 -10.0..."
7,0,Mua,"POLYGON ((142.26095 -10.12127, 142.26047 -10.1..."
8,0,Warraber,"POLYGON ((142.82937 -10.20402, 142.83078 -10.2..."
9,0,Poruma,"POLYGON ((143.06883 -10.04832, 143.07186 -10.0..."


In [249]:
pwd

'/home/jovyan/dev/dea-notebooks/Testing'

In [250]:
# ## Original: single polygon load

# # Load data from WFS for study area bounding box
# bbox = study_area.geometry.bounds.values[0]
# # bbox = study_area.geometry.bounds#.values[0]
# ratesofchange_gdf = get_coastlines(
    
#     layer="rates_of_change",
#     bbox=bbox
# )

# # Clip returned data to polygon extent
# ratesofchange_gdf = gpd.clip(ratesofchange_gdf, mask=study_area.to_crs("EPSG:3577"))
# ratesofchange_gdf

In [251]:
from shapely.geometry import box

In [252]:
# Load archived data as a geopandas.GeoDataFrame
# coastlines_gdf = gpd.read_file("coastlines_v2.0.0.shp/coastlines_v2.0.0_rates_of_change.shp")

bl_year = 2025

path = f'../../dea-coastlines/data/processed/development_TSRA_{bl_year}_bl/coastlines_development_TSRA_{bl_year}_bl.gpkg'

## Multi-polygon load

study_area_albers = study_area.to_crs("EPSG:3577")

ratesofchange1 = {}

for x in range(0, len(study_area)):

    key = study_area.iloc[x]['name']
    print(f'Polygon {x +1} of {len(study_area)}, {key}, is processing')
    
    # Load data from coastlines reanalysis for study area bounding box
    bbox = study_area.geometry.bounds.values[x]
    # bbox = study_area.geometry.bounds#.values[0]
    ratesofchange_gdf = get_coastlines1( 
        response=path,
        layer="rates_of_change",
        bbox=tuple(bbox) 
    )
    # Clip returned data to polygon extent
    # ratesofchange_gdf = gpd.clip(ratesofchange_gdf, mask=study_area_albers.iloc[x].geometry)#.to_crs("EPSG:3577"))
    # ratesofchange_gdf = coastlines_gdf.clip(study_area_albers.iloc[x].geometry.bounds,study_area_albers.crs)
    
    # Add to dictionary
    value = ratesofchange_gdf
    ratesofchange1[key] = value


# Drop polygons with no rate change points
ratesofchange = {island: df for island, df in ratesofchange1.items() 
                 # if 'uid' in df.columns}
                 if len(df)>0}

dropped = []
for key in ratesofchange1.keys():
    if key not in ratesofchange.keys():
        dropped.append(key)
print ('-----')
print (f'Excluded polgyons due to absence of change rate data: {dropped}')
    

Polygon 1 of 17, Boigu, is processing
Polygon 2 of 17, Saibai, is processing
Polygon 3 of 17, Dauan, is processing
Polygon 4 of 17, Waral Kawa, is processing
Polygon 5 of 17, Awail Kawa, is processing
Polygon 6 of 17, Mabuiag, is processing
Polygon 7 of 17, Badu, is processing
Polygon 8 of 17, Mua, is processing
Polygon 9 of 17, Warraber, is processing
Polygon 10 of 17, Poruma, is processing
Polygon 11 of 17, Iama, is processing
Polygon 12 of 17, Masig, is processing
Polygon 13 of 17, Ugar, is processing
Polygon 14 of 17, Erub, is processing
Polygon 15 of 17, Mer, is processing
Polygon 16 of 17, Maizab Kaur, is processing
Polygon 17 of 17, Torres Strait, is processing
-----
Excluded polgyons due to absence of change rate data: ['Waral Kawa', 'Awail Kawa', 'Mabuiag', 'Badu', 'Maizab Kaur']


In [253]:
# ## Multi-polygon load

# study_area_albers = study_area.to_crs("EPSG:3577")

# ratesofchange1 = {}

# for x in range(0, len(study_area)):

#     key = study_area.iloc[x]['name']
#     print(f'Polygon {x +1} of {len(study_area)}, {key}, is processing')
    
#     # Load data from WFS for study area bounding box
#     bbox = study_area.geometry.bounds.values[x]
#     # bbox = study_area.geometry.bounds#.values[0]
#     ratesofchange_gdf = get_coastlines( # Use for latest WFS DEA Coastlines data
#         bbox=bbox , layer="rates_of_change"
#     )
    
#     # Clip returned data to polygon extent
#     ratesofchange_gdf = gpd.clip(ratesofchange_gdf, mask=study_area_albers.iloc[x].geometry)#.to_crs("EPSG:3577"))

#     # Add to dictionary
#     value = ratesofchange_gdf
#     ratesofchange1[key] = value


# # Drop polygons with no rate change points
# ratesofchange = {island: df for island, df in ratesofchange1.items() 
#                  if 'id' in df.columns}
# dropped = []
# for key in ratesofchange1.keys():
#     if key not in ratesofchange.keys():
#         dropped.append(key)
# print ('-----')
# print (f'Excluded polgyons due to absence of change rate data: {dropped}')
    

## Calculate long term site change rates

In [254]:
# ##Calculate long term change rates for a given baseline year for a single polygon

# # Setting a 'start' date is optional. If not set, it will default to 1988
# # start = 1988
# baseline = 2024
# start_years=[2019, 1988]

# change_epochs = {}

# for year in start_years:
#     print (year)
#     key = f'{year}-{baseline}'
#     value = calculate_regressions(ratesofchange_gdf,
#                          year, baseline)
#     change_epochs[key] = value


In [255]:
bl_year-5

2020

In [256]:
##Calculate long term change rates for a given baseline year for multiple polygons

# Setting a 'start' date is optional. If not set, it will default to 1988
# start = 1988
baseline = bl_year
start_years=[bl_year-4, 1988]

change_epochs = {}
# change_rates = {}

for year in start_years:
    key1 = f'{year}-{baseline}' 
    change_epochs[key1]={}
    for island in list(ratesofchange.keys()):
        if len(ratesofchange[island])>0:
            print (f'{year}, {island}')
    
            change_epochs[key1][f'{island}'] = calculate_regressions(ratesofchange[island],
                                 year, baseline)
            
    
    # value2 = calculate_regressions(ratesofchange_gdf,
    #                      year, baseline)
    # change_epochs[key1] = change_rates



2021, Boigu
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')
2021, Saibai
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')
2021, Dauan
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')
2021, Mua
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')
2021, Warraber
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')
2021, Poruma
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')
2021, Iama
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')
2021, Masig
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')
2021, Ugar
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')
2021, Erub
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype=

## Short term change

Need to run the whole coastlines workflow over the area of interest from scratch e.g. https://github.com/GeoscienceAustralia/dea-coastlines/blob/stable/notebooks/DEACoastlines_generation_vector.ipynb
This will allow me to set new baselines from which to calculate other SoE epochs (e.g. 1988 to 2020, 1988 to 2015) of both short (5 year) and long term change

The following cells are temporarily commented out but are live code, requiring `load_rasters` func from https://github.com/GeoscienceAustralia/dea-coastlines/blob/stable/notebooks/DEACoastlines_generation_vector.ipynb to work.

In [257]:
# # For short term change use below from https://github.com/GeoscienceAustralia/dea-coastlines/blob/155ef969de9f1126d82d5ee5a7f03c942ecae118/coastlines/vector.py#L795

# def points_on_line(gdf, index, distance=30):
#     """
#     Generates evenly-spaced point features along a specific line feature
#     in a `geopandas.GeoDataFrame`.

#     Parameters:
#     -----------
#     gdf : geopandas.GeoDataFrame
#         A `geopandas.GeoDataFrame` containing line features with an
#         index and CRS.
#     index : string or int
#         An value giving the index of the line to generate points along
#     distance : integer or float, optional
#         A number giving the interval at which to generate points along
#         the line feature. Defaults to 30, which will generate a point
#         at every 30 metres along the line.

#     Returns:
#     --------
#     points_gdf : geopandas.GeoDataFrame
#         A `geopandas.GeoDataFrame` containing point features at every
#         `distance` along the selected line.

#     """

#     # Select individual line to generate points along
#     line_feature = gdf.loc[[index]].geometry

#     # If multiple features are returned, take unary union
#     if line_feature.shape[0] > 0:
#         line_feature = line_feature.unary_union
#     else:
#         line_feature = line_feature.iloc[0]

#     # Generate points along line and convert to geopandas.GeoDataFrame
#     points_line = [
#         line_feature.interpolate(i)
#         for i in range(0, int(line_feature.length), distance)
#     ]
#     points_gdf = gpd.GeoDataFrame(geometry=points_line, crs=gdf.crs)

#     return points_gdf

# def annual_movements(
#     points_gdf, contours_gdf, yearly_ds, baseline_year, water_index, max_valid_dist=1000
# ):
#     """
#     For each rate of change point along the baseline annual coastline,
#     compute the distance to the nearest point on all neighbouring annual
#     coastlines and add this data as new fields in the dataset.

#     Distances are assigned a directionality (negative = located inland,
#     positive = located sea-ward) by sampling water index values from the
#     underlying DEA Coastlines rasters to determine if a coastline was
#     located in wetter or drier terrain than the baseline coastline.

#     Parameters:
#     -----------
#     points_gdf : geopandas.GeoDataFrame
#         A `geopandas.GeoDataFrame` containing rates of change points.
#     contours_gdf : geopandas.GeoDataFrame
#         A `geopandas.GeoDataFrame` containing annual coastlines.
#     yearly_ds : xarray.Dataset
#         An `xarray.Dataset` containing annual DEA CoastLines rasters.
#     baseline_year : string
#         A string giving the year used as the baseline when generating
#         the rates of change points dataset. This is used to load DEA
#         CoastLines water index rasters to calculate change
#         directionality.
#     water_index : string
#         A string giving the water index used in the analysis. This is
#         used to load DEA CoastLines water index rasters to calculate
#         change directionality.
#     max_valid_dist : int or float, optional
#         Any annual distance greater than this distance will be set
#         to `np.nan`.

#     Returns:
#     --------
#     points_gdf : geopandas.GeoDataFrame
#         A `geopandas.GeoDataFrame` containing rates of change points
#         with added 'dist_*' attribute columns giving the distance to
#         each annual coastline from the baseline. Negative values
#         indicate that an annual coastline was located inland of the
#         baseline; positive values indicate the coastline was located
#         towards the ocean.
#     """

#     def _point_interp(points, array, **kwargs):
#         points_gs = gpd.GeoSeries(points)
#         x_vals = xr.DataArray(points_gs.x, dims="z")
#         y_vals = xr.DataArray(points_gs.y, dims="z")

#         return array.interp(x=x_vals, y=y_vals, **kwargs)

#     # Get array of water index values for baseline time period
#     baseline_array = yearly_ds[water_index].sel(year=int(baseline_year))

#     # Copy baseline point geometry to new column in points dataset
#     points_gdf["p_baseline"] = points_gdf.geometry

#     # Years to analyse
#     years = contours_gdf.index.unique().values

#     # Iterate through all comparison years in contour gdf
#     for comp_year in years:
#         # Set comparison contour
#         comp_contour = contours_gdf.loc[[comp_year]].geometry.iloc[0]

#         # Find nearest point on comparison contour, and add these to points dataset
#         points_gdf[f"p_{comp_year}"] = points_gdf.apply(
#             lambda x: nearest_points(x.p_baseline, comp_contour)[1], axis=1
#         )

#         # Compute distance between baseline and comparison year points and add
#         # this distance as a new field named by the current year being analysed
#         distances = points_gdf.apply(
#             lambda x: x.geometry.distance(x[f"p_{comp_year}"]), axis=1
#         )

#         # Set any value over X m to NaN
#         points_gdf[f"dist_{comp_year}"] = distances.where(distances < max_valid_dist)

#         # Extract comparison array containing water index values for the
#         # current year being analysed
#         comp_array = yearly_ds[water_index].sel(year=int(comp_year))

#         # Sample water index values for baseline and comparison points
#         points_gdf["index_comp_p1"] = _point_interp(points_gdf.geometry, comp_array)
#         points_gdf["index_baseline_p2"] = _point_interp(
#             points_gdf[f"p_{comp_year}"], baseline_array
#         )

#         # Compute change directionality (positive = located towards the
#         # ocean; negative = located inland)
#         points_gdf["loss_gain"] = np.where(
#             points_gdf.index_baseline_p2 > points_gdf.index_comp_p1, 1, -1
#         )

#         # Ensure NaNs are correctly propagated (otherwise, X > NaN
#         # will return False, resulting in an incorrect land-ward direction)
#         is_nan = points_gdf[["index_comp_p1", "index_baseline_p2"]].isna().any(axis=1)
#         points_gdf["loss_gain"] = points_gdf["loss_gain"].where(~is_nan)

#         # Multiply distance to set change to negative, positive or NaN
#         points_gdf[f"dist_{comp_year}"] = (
#             points_gdf[f"dist_{comp_year}"] * points_gdf.loss_gain
#         )

#         # Calculate compass bearing from baseline to comparison point;
#         # first we need our points in lat-lon
#         lat_lon = points_gdf[["p_baseline", f"p_{comp_year}"]].apply(
#             lambda x: gpd.GeoSeries(x, crs=points_gdf.crs).to_crs("EPSG:4326")
#         )

#         geodesic = pyproj.Geod(ellps="WGS84")
#         bearings = geodesic.inv(
#             lons1=lat_lon.iloc[:, 0].values.x,
#             lats1=lat_lon.iloc[:, 0].values.y,
#             lons2=lat_lon.iloc[:, 1].values.x,
#             lats2=lat_lon.iloc[:, 1].values.y,
#         )[0]

#         # Add bearing as a new column after first restricting
#         # angles between 0 and 180 as we are only interested in
#         # the overall axis of our points e.g. north-south
#         points_gdf[f"bearings_{comp_year}"] = bearings % 180

#     # Calculate mean and standard deviation of angles
#     points_gdf["angle_mean"] = (
#         points_gdf.loc[:, points_gdf.columns.str.contains("bearings_")]
#         .apply(lambda x: circmean(x, high=180), axis=1)
#         .round(0)
#         .astype(int)
#     )
#     points_gdf["angle_std"] = (
#         points_gdf.loc[:, points_gdf.columns.str.contains("bearings_")]
#         .apply(lambda x: circstd(x, high=180), axis=1)
#         .round(0)
#         .astype(int)
#     )

#     # Keep only required columns
#     to_keep = points_gdf.columns.str.contains("dist|geometry|angle")
#     points_gdf = points_gdf.loc[:, to_keep]
#     points_gdf = points_gdf.assign(**{f"dist_{baseline_year}": 0.0})
#     points_gdf = points_gdf.round(2)

#     return points_gdf

In [258]:
# # Load data from WFS for study area bounding box
# bbox = study_area.geometry.bounds.values[0]
# # bbox = study_area.geometry.bounds#.values[0]
# shorelines_gdf = get_coastlines(
#     bbox=bbox, layer="shorelines_annual"
# )

# # Clip returned data to polygon extent
# shorelines_gdf = gpd.clip(shorelines_gdf, mask=study_area.to_crs("EPSG:3577"))

# # Reset the index for vector calculation
# shorelines_gdf.set_index('year', inplace=True)

# shorelines_gdf

In [259]:
# shorelines_gdf.plot()

# Compute statistics
## Create stats points on baseline shoreline

In [260]:
# # Extract statistics modelling points along baseline shoreline

# # start_year = 1988
# # end_year = 2024
# baseline_year = 2024

# try:
#     points_gdf = points_on_line(shorelines_gdf, baseline_year, distance=30)

# except KeyError:
#     points_gdf = None

# print(points_gdf)

## Measure annual coastline movements

In [261]:
# if points_gdf is not None and len(points_gdf) > 0:
    
#     # Calculate annual movements for every shoreline
#     # compared to the baseline year
#     points_gdf = annual_movements(
#         points_gdf,
#         shorelines_gdf,
#         yearly_ds,
#         baseline_year,
#         water_index,
#         max_valid_dist=1200,
#     )
    
#     # Reindex to add any missing annual columns to the dataset
#     points_gdf = points_gdf.reindex(
#         columns=[
#             "geometry",
#             *[f"dist_{i}" for i in range(start_year, end_year + 1)],
#             "angle_mean",
#             "angle_std",
#         ]
#     )

# points_gdf

In [262]:
# key = change_epochs['2019-2024'].copy()
# key['certainty']=ratesofchange_gdf['certainty']

In [263]:
# print(len(key.query("certainty == 'good'")))
# # print(len(ratesofchange_gdf))
# # print(len(key))
# print (len(ratesofchange_gdf.query("certainty == 'good'")))

## Tabularise data

In [264]:
# # Code for single polygon analysis
# for key in list(change_epochs.keys()):

#     # A one-off only to reattach the quality data to the current all-time dataset.
#     # REMOVE THIS LINE WHEN OTHER EPOCHS ARE CALCULATED UNLESS THEIR SPECIFIC
#     # QUALITY DATA NEEDS REATTACHING TOO FROM THE LONG TERM DATASET

#     change_epochs[key]['certainty']=ratesofchange_gdf['certainty']
    
#     # Optional: Keep only rates of change points with "good" certainty 
#     # (i.e. no poor quality flags)
#     change_epochs[key] = change_epochs[key].query("certainty == 'good'").copy()
      
#     # Get the column names
#     rate_cols = change_epochs[key].columns[change_epochs[key].columns.str.startswith('rate_time')]
#     sig_cols = change_epochs[key].columns[change_epochs[key].columns.str.startswith('sig_time')]    

#     # Optional: Apply correction factor from Bishop-Taylor et al. 2021
#     change_epochs[key].loc[:, rate_cols] = change_epochs[key].loc[:, rate_cols] + 0.08
        
#     # Add x and y coords to data
#     change_epochs[key]["y_coord"] = change_epochs[key].geometry.y
#     change_epochs[key]["x_coord"] = change_epochs[key].geometry.x
    
#     # Create filtered columns (copy of rate_time columns)
#     for rate_col in rate_cols:
#         filtered_col_name = f'{rate_col}_sig_filtered'
#         change_epochs[key][filtered_col_name] = change_epochs[key][rate_col].copy()
    
#     # Replace with 0 where ANY sig_time column > 0.01
#     for sig_col in sig_cols:
#         for rate_col in rate_cols:
#             filtered_col_name = f'{rate_col}_sig_filtered'
#             change_epochs[key].loc[change_epochs[key][sig_col] > 0.01, filtered_col_name] = 0


In [265]:
# Code for multi polygon analysis
for key in change_epochs.keys():
    for key1 in change_epochs[key].keys():

        # A one-off only to reattach the quality data to the current all-time dataset.
        # REMOVE THIS LINE WHEN OTHER EPOCHS ARE CALCULATED UNLESS THEIR SPECIFIC
        # QUALITY DATA NEEDS REATTACHING TOO FROM THE LONG TERM DATASET
    
        change_epochs[key][key1]['certainty']=ratesofchange[key1]['certainty']
        
        # Optional: Keep only rates of change points with "good" certainty 
        # (i.e. no poor quality flags)
        change_epochs[key][key1] = change_epochs[key][key1].query("certainty == 'good'").copy()
          
        # Get the column names
        rate_cols = change_epochs[key][key1].columns[change_epochs[key][key1].columns.str.startswith('rate_time')]
        sig_cols = change_epochs[key][key1].columns[change_epochs[key][key1].columns.str.startswith('sig_time')]    

        # print(rate_cols)
        
        # Optional: Apply correction factor from Bishop-Taylor et al. 2021
        change_epochs[key][key1].loc[:, rate_cols] = change_epochs[key][key1].loc[:, rate_cols] + 0.08
            
        # Add x and y coords to data
        change_epochs[key][key1]["y_coord"] = change_epochs[key][key1].geometry.y
        change_epochs[key][key1]["x_coord"] = change_epochs[key][key1].geometry.x
        
        # Create filtered columns (copy of rate_time columns)
        for rate_col in rate_cols:
            filtered_col_name = f'{rate_col}_sig_filtered'
            change_epochs[key][key1][filtered_col_name] = change_epochs[key][key1][rate_col].copy()
        
        # Replace with 0 where ANY sig_time column > 0.01
        for sig_col in sig_cols:
            for rate_col in rate_cols:
                filtered_col_name = f'{rate_col}_sig_filtered'
                change_epochs[key][key1].loc[change_epochs[key][key1][sig_col] > 0.01, filtered_col_name] = 0

# Drop columns with no "good" certainty rates of change
pop = []

for key in change_epochs.keys():
    for key1 in change_epochs[key].keys():
        if len(change_epochs[key][key1])==0:
            pop.append((key,key1))
            print(f'{key1} {key} removed due to low certainty rate change points')
for k in pop:
    change_epochs[k[0]].pop(k[-1],None)
# # Print remaining keys
# for key in change_epochs.keys():
#     print (key)
#     for key1 in change_epochs[key].keys():
#         print(key1)


Boigu 2021-2025 removed due to low certainty rate change points
Boigu 1988-2025 removed due to low certainty rate change points


In [266]:
# ratesofchange_standardised_gdf = {}

# for key in list(change_epochs.keys()):

#     # Resample to make sure we have evenly spaced rows (important for sensible rolling mean)
#     bin_size = 30
#     min_coord = change_epochs[key]["y_coord"].min()
#     max_coord = change_epochs[key]["y_coord"].max()
#     bin_edges = np.arange(min_coord, max_coord, bin_size)
#     groups = pd.cut(
#         change_epochs[key]["y_coord"],
#         bins=bin_edges,
#         labels=bin_edges[:-1] + (bin_size / 2),
#         right=False,
#     )
#     # ratesofchange_standardised_gdf = ratesofchange_gdf.groupby(groups).mean() ## Original code
#     ratesofchange_standardised_gdf[key] = change_epochs[key].groupby(groups, observed=True).mean(numeric_only=True) ## TEMPORARY CHANGE FOR TESTING 21/09/23 CP
    
#     # Set index to numeric so we can plot it nicely
#     ratesofchange_standardised_gdf[key].index = pd.to_numeric(
#         ratesofchange_standardised_gdf[key].index
#     )

## Rolling mean

In [267]:
# ratesofchange_rolling_gdf={}

# for key in list(ratesofchange_standardised_gdf.keys()):

#     # Apply rolling median
#     window_size = 500  # km
#     window_n = int(window_size / 30)
#     ratesofchange_rolling_gdf[key] = ratesofchange_standardised_gdf[key].rolling(
#         window=window_n, center=True, min_periods=1
#     ).mean()
    
#     # Apply an additional level of aesthetic smoothing (can be removed)
#     ratesofchange_rolling_gdf[key] = ratesofchange_rolling_gdf[key].rolling(
#         window=window_n, center=True, min_periods=1
#     ).mean()

## Plotting

In [268]:
# ## Set the fontsize for the plot
# plt.rcParams.update({'font.size': 12})

# # Common parameters for all-time and epoch plots
# # all_time = [key for key in ratesofchange_rolling_gdf.keys() if '1988' in key]



# for key in list(ratesofchange_rolling_gdf.keys()):
#     # Set first and last entry to zero so we get clean graph outlines
#     indexfirst = ratesofchange_rolling_gdf[key].index.min()
#     indexlast = ratesofchange_rolling_gdf[key].index.max()
    
#     ## All time plotting - statistically significant points only (sig_time <0.01) and good quality data (certainty = “good”)
#     ratesofchange_rolling_gdf[key].loc[indexfirst,f"rate_time_{key}_sig_filtered"] = 0
#     ratesofchange_rolling_gdf[key].loc[indexlast,f"rate_time_{key}_sig_filtered"] = 0
    
#     # Split out positive and negative rates so we can plot them individually
#     ratesofchange_positive_gdf = ratesofchange_rolling_gdf[key][f"rate_time_{key}_sig_filtered"].clip(0, np.inf)
#     ratesofchange_negative_gdf = ratesofchange_rolling_gdf[key][f"rate_time_{key}_sig_filtered"].clip(-np.inf, 0)
    
#     # Plot negative rates in red, positive in blue
#     fig, ax = plt.subplots(figsize=(2, 11.5))#7))#1.75))#5))#7))
#     ax.fill_betweenx(
#         ratesofchange_positive_gdf.index, 0, ratesofchange_positive_gdf, color="#6caed1"
#     )
#     ax.fill_betweenx(
#         ratesofchange_negative_gdf.index, ratesofchange_negative_gdf, 0, color="#eb7668"
#     )
#     ax.plot(
#         ratesofchange_rolling_gdf[key][f"rate_time_{key}_sig_filtered"],
#         ratesofchange_rolling_gdf[key].index,
#         color="black",
#         linewidth=1
#     )
    
#     # Add vertical axis line
#     ax.axvline(0, color="black", linewidth=1)

#     # Styling
#     ax.set_xlim(-1.33, 1.33)
#     ax.set_ylim(min_coord, max_coord)
#     ax.set_xticks(ticks=[-1, 0, 1])
#     ax.set_xlabel(f"(m / year between \n {key}")
#     ax.spines[["top", "left", "right"]].set_visible(False)
#     ax.tick_params(top=False, left=False, labelleft=False, labeltop=False)
#     # ax.text
    
#     # Export
#     fig.savefig(f"/home/jovyan/dev/dea-notebooks/Testing/Torres_Strait_{key}_latsummary.svg", bbox_inches="tight", transparent=False)

## Statistical comparison

In [269]:
# region = 'Torres_Strait'

In [270]:
# # Indentify short and long epochs
# long_term = [key for key in change_epochs.keys() if '1988' in key]
# short_term = [key for key in change_epochs.keys() if '1988' not in key]

# ## Baseline comparison stats for mainland polygons
# v1 = change_epochs[long_term[0]][f'rate_time_{long_term[0]}_sig_filtered'].values
# v2 = change_epochs[short_term[0]][f'rate_time_{short_term[0]}_sig_filtered'].values

# rel = ttest_rel(v1,v2)
# # rel

# ratetimemean = change_epochs[long_term[0]][f'rate_time_{long_term[0]}_sig_filtered'].mean().round(2)
# ratetimestd = round(change_epochs[long_term[0]][f'rate_time_{long_term[0]}_sig_filtered'].std(),2)
# epochmean = change_epochs[short_term[0]][f'rate_time_{short_term[0]}_sig_filtered'].mean().round(2)
# epochstd = round(change_epochs[short_term[0]][f'rate_time_{short_term[0]}_sig_filtered'].std(),2)

# # Run once as the master dataframe
# data = {
#         long_term[0]:f'{ratetimemean} ({ratetimestd})', 
#         short_term[0]:f'{epochmean} ({epochstd})',
#         f'pvalue ({long_term[0]} vs {short_term[0]})':rel.pvalue,
#         f'tstat ({long_term[0]} vs {short_term[0]})':round(rel.statistic,2),
#         f'df ({long_term[0]} vs {short_term[0]})':rel.df
#         }
# StatSummary = pd.DataFrame(data, index=[region]) 

In [271]:
# # Append new row to dataframe
# StatSummary.loc[len(StatSummary.index)] =          [f'{ratetimemean} ({ratetimestd})', 
#                                                    f'{epochmean} ({epochstd})',
#                                                    rel.pvalue.round(2),
#                                                    round(rel.statistic,2),
#                                                    rel.df]
# ## Update index label
# StatSummary.rename(index={StatSummary.loc[len(StatSummary.index)-1].name:region},inplace=True)

In [272]:
# StatSummary.to_csv("/home/jovyan/dev/dea-notebooks/Testing/StatSummary.csv")

# After: https://gist.github.com/robbibt/760dcf367be4b98c493e70dd577aca6a

In [273]:
def change_summary(df, sig=0.01, rate=0.30, bias=0.08):

    # Create booleans indicating whether points were significant
    # or greater than the minimum accuracy of the method
    sig_bool = df.sig_time <= sig
    # rate_bool = (df.rate_time + bias).abs() >= rate #CP removed as the bias correction has already been applied
    rate_bool = (df.rate_time).abs() >= rate

    # Calculate dynamic % (sig points greater than min rate)
    stat_dict = {}
    stat_dict['dynamic'] = (sig_bool & rate_bool).mean()

    # Calculate stable % (non-sig points or less than min rate)
    stat_dict['stable'] = 1.0 - stat_dict['dynamic']

    # For each rate of change categoru, calculate percent greater
    # (prograding) or percent smaller (eroding coasts)

    for rate_cat in [0.0, 0.5, 1.0, 3.0, 5.0]:
        stat_dict[f'eroding_{rate_cat}'] = (
            # sig_bool & rate_bool & (df.rate_time + bias < -rate_cat)).mean() #CP removed as the bias correction has already been applied
            sig_bool & rate_bool & (df.rate_time < -rate_cat)).mean()
        stat_dict[f'prograd_{rate_cat}'] = (
            # sig_bool & rate_bool & (df.rate_time + bias > rate_cat)).mean() #CP removed as the bias correction has already been applied
            sig_bool & rate_bool & (df.rate_time > rate_cat)).mean()

    return pd.Series(stat_dict)

In [274]:
# # Long term change for a single polygon summary

# summary_df = pd.DataFrame()
# for key in list(change_epochs.keys()):
#     # Load point data and Coastal Compartment regions data
#     # coastlines_data = gpd.read_file('../releases/DEACoastlines_v1.0.0/Shapefile/DEACoastlines_ratesofchange_v1.0.0.shp')[['rate_time', 'sig_time', 'geometry']]
#     coastlines_data = change_epochs[key][[f'rate_time_{key}', f'sig_time_{key}', 'geometry']]
#     coastlines_data = coastlines_data.rename(columns={f'rate_time_{key}' : 'rate_time'})
#     coastlines_data = coastlines_data.rename(columns={f'sig_time_{key}' : 'sig_time'})
    
    
#     # For the single Torres Strait polygon:
    
#     # Compute summaries of change for all regions
#     # summary_df = coastlines_data.apply(lambda x: change_summary(x)).T
#     summary_df[key] = change_summary(df=coastlines_data, sig=0.01)

# # # Sort into pretty format
# summary_df = summary_df.loc[[
#     'dynamic', 'stable', 'eroding_0.0', 'eroding_0.5', 'eroding_1.0',
#     'eroding_3.0', 'eroding_5.0', 'prograd_0.0', 'prograd_0.5', 'prograd_1.0',
#     'prograd_3.0', 'prograd_5.0'
# ]]

# # Rename index
# summary_df.index = ['Dynamic', 'Stable', 
#                     'Eroding',     
#                     '    > 0.5 m / year', '    > 1.0 m / year', 
#                     '    > 3.0 m / year', '    > 5.0 m / year', 
#                     'Prograding', 
#                     '    > 0.5 m / year', '    > 1.0 m / year', 
#                     '    > 3.0 m / year', '    > 5.0 m / year']

# # Scale and round
# summary_df = np.round((summary_df * 100),2)

# # summary_df.columns=[long_term[0]]

# summary_df

In [275]:
# Long and short summaries for multiple polgyons

summary_df = pd.DataFrame()
for key in change_epochs.keys():
    for key1 in change_epochs[key].keys():
        # Load point data and Coastal Compartment regions data
        # coastlines_data = gpd.read_file('../releases/DEACoastlines_v1.0.0/Shapefile/DEACoastlines_ratesofchange_v1.0.0.shp')[['rate_time', 'sig_time', 'geometry']]
        coastlines_data = change_epochs[key][key1][[f'rate_time_{key}', f'sig_time_{key}', 'geometry']]
        coastlines_data = coastlines_data.rename(columns={f'rate_time_{key}' : 'rate_time'})
        coastlines_data = coastlines_data.rename(columns={f'sig_time_{key}' : 'sig_time'})
        
        
        # For the single Torres Strait polygon:
        
        # Compute summaries of change for all regions
        # summary_df = coastlines_data.apply(lambda x: change_summary(x)).T
        summary_df[f'{key1} {key}'] = change_summary(df=coastlines_data, sig=0.01)

# # Sort into pretty format
summary_df = summary_df.loc[[
    'dynamic', 'stable', 'eroding_0.0', 'eroding_0.5', 'eroding_1.0',
    'eroding_3.0', 'eroding_5.0', 'prograd_0.0', 'prograd_0.5', 'prograd_1.0',
    'prograd_3.0', 'prograd_5.0'
]]

# Rename index
summary_df.index = ['Dynamic', 'Stable', 
                    'Eroding',     
                    '    > 0.5 m / year', '    > 1.0 m / year', 
                    '    > 3.0 m / year', '    > 5.0 m / year', 
                    'Prograding', 
                    '    > 0.5 m / year', '    > 1.0 m / year', 
                    '    > 3.0 m / year', '    > 5.0 m / year']

# Scale and round
summary_df = np.round((summary_df * 100),2)

# reorder column names
gross = [f'Torres Strait {start_years[-1]}-{baseline}',
       f'Torres Strait {start_years[0]}-{baseline}']
sort = list(summary_df.columns.sort_values())
for val in gross:
    if val in sort:
        sort.remove(val)

summary_df=summary_df[gross+sort]
summary_df

summary_df

,Torres Strait 1988-2025,Torres Strait 2021-2025,Dauan 1988-2025,Dauan 2021-2025,Erub 1988-2025,Erub 2021-2025,Iama 1988-2025,Iama 2021-2025,Masig 1988-2025,Masig 2021-2025,...,Mua 1988-2025,Mua 2021-2025,Poruma 1988-2025,Poruma 2021-2025,Saibai 1988-2025,Saibai 2021-2025,Ugar 1988-2025,Ugar 2021-2025,Warraber 1988-2025,Warraber 2021-2025
Dynamic,32.18,4.04,36.89,0.32,5.14,2.43,42.17,4.42,54.98,0.95,...,20.07,4.70,15.27,0.76,39.08,7.46,0.0,3.66,40.68,1.69
Stable,67.82,95.96,63.11,99.68,94.86,97.57,57.83,95.58,45.02,99.05,...,79.93,95.30,84.73,99.24,60.92,92.54,100.0,96.34,59.32,98.31
Eroding,13.34,0.44,27.18,0.00,4.59,1.62,5.22,0.40,24.17,0.95,...,2.02,0.67,5.34,0.76,28.18,0.12,0.0,0.00,21.19,0.00
> 0.5 m / year,7.65,0.41,7.77,0.00,0.00,1.62,1.20,0.40,13.27,0.95,...,1.18,0.59,3.82,0.76,18.77,0.12,0.0,0.00,15.25,0.00
> 1.0 m / year,1.60,0.36,0.32,0.00,0.00,1.62,0.00,0.40,6.16,0.95,...,0.17,0.42,3.05,0.76,4.03,0.12,0.0,0.00,5.08,0.00
> 3.0 m / year,0.00,0.21,0.00,0.00,0.00,1.08,0.00,0.00,0.00,0.95,...,0.00,0.42,0.00,0.00,0.00,0.06,0.0,0.00,0.00,0.00
> 5.0 m / year,0.00,0.10,0.00,0.00,0.00,0.54,0.00,0.00,0.00,0.00,...,0.00,0.17,0.00,0.00,0.00,0.06,0.0,0.00,0.00,0.00
Prograding,18.84,3.60,9.71,0.32,0.54,0.81,36.95,4.02,30.81,0.00,...,18.05,4.03,9.92,0.00,10.89,7.34,0.0,3.66,19.49,1.69
> 0.5 m / year,12.83,3.57,1.29,0.32,0.00,0.81,30.52,4.02,19.43,0.00,...,7.89,4.03,8.40,0.00,6.34,7.34,0.0,2.44,17.80,1.69
> 1.0 m / year,3.81,3.49,0.00,0.32,0.00,0.81,23.29,4.02,6.16,0.00,...,1.26,3.86,4.58,0.00,1.60,7.28,0.0,2.44,9.32,1.69


In [276]:
summary_df.to_csv(f"/home/jovyan/dev/dea-notebooks/Testing/TS_Islands_{baseline}_baseline_long_and_short_epochs.csv")